# STRATA, precisão sob demanda, E6: o pacote inteiro num 8B (Llama-3.1-8B-Instruct)

Roda no **Colab** ou no **Kaggle**; o notebook descobre onde está. A GPU só é usada na conferência bit a bit; a geração roda na CPU.

- **Colab:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*. Leva cerca de 1 hora. No fim, baixa três arquivos.
- **Kaggle:** *File → Import Notebook* com este arquivo. No painel da direita (*Session options*), escolha **Accelerator: GPU T4 x2** (a P100 pode não funcionar com o PyTorch atual) e ligue **Internet** (precisa de conta com telefone verificado). Depois *Run All*. Leva umas 2 a 3 horas, porque o Kaggle tem 4 núcleos de CPU. No fim, os três arquivos ficam em `/kaggle/working`, na aba *Output* do painel da direita.

Os três arquivos são um `.md`, um `.json` e um `.zip` com os CSV e as respostas. Cole-os na thread.

**O que é testado (E6 de `docs/08-precisao-sob-demanda.md`).**
1. O modelo começa no **Q4_K_M do Hub** (bartowski), mapeado do disco sem mudar um byte, e responde na hora.
2. Os planos dos níveis 1 e 2 chegam nos **arquivos de estrato reais** da distribuição B.10 (`strata-ref distribui`, `e01` e `e02`). São servidos por HTTP com pedidos de faixa e um teto de banda (100 e 300 Mbit/s) e baixados pelo `baixa_estratos.py --ate-nivel 2` enquanto o modelo gera.
3. Cada plano tem o **SHA-256 conferido** contra o índice antes de entrar no modelo e é preparado na própria geração (até 4 ms entre dois tokens). As matrizes sobem para o nível 1 e depois para o 2, sem recarregar.
4. Perto do fim, o modelo **desce** para o nível 1 e depois para o 0, entre dois tokens.
5. **Conferência bit a bit:** os logits do modelo que subiu são iguais aos dos GGUFs de nível 1 e 2 gravados do zero por outro caminho (`strata_gguf_niveis.py`, Python, na GPU).

**Regra escrita antes:**
- **Funciona de ponta a ponta** se:
  - os logits são iguais bit a bit nos níveis 0, 1 e 2, inclusive depois de descer;
  - os dois níveis completam com os planos conferidos;
  - a resposta continua coerente através das trocas.
- **A pausa** é comparada com um controle: o mesmo download, sem subida, na mesma faixa de tokens. "Não trava" quer dizer que o p99 e o pior token ficam no máximo 1,5× os do controle.

**Por que um thread a menos durante o download:** no 1B, num container de 4 núcleos, o download sozinho (sem subida) já travava a geração (p99 de 130 a 1.800 ms) com as 4 threads; com 3 enquanto ele corre, o controle e a subida ficaram em 68–90 ms. Custa ~20% de velocidade só durante o download.

**Quantos tokens:** cada rodada gera o suficiente para cobrir o download dos dois níveis, calculado a partir da velocidade medida na conferência. Numa máquina lenta, a rodada fica mais longa, mas não corta a subida no meio.

**Limite conhecido:** o servidor roda na mesma máquina (rede simulada por teto de banda, sem perda nem latência variável), e disputa CPU com a geração.

In [ ]:
MODELO_HF = "unsloth/Meta-Llama-3.1-8B-Instruct"                     # os pesos BF16 (para gravar os planos)
GGUF_REPO = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
GGUF_ARQ = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"                   # o nível 0: o arquivo publicado, sem mudar
MBITS = [100, 300]         # tetos de banda do servidor local
N_TOKENS = {}              # tokens por rodada; vazio = calcula pela velocidade medida (cobre o download com folga)
INICIO = 16                # a subida começa neste token
T_SUBIDA = -1              # threads da geração enquanto o download corre (-1 = um a menos que THREADS; 0 = as mesmas)
CONFERIR_BIT_A_BIT = True  # precisa de GPU (os GGUFs de nível 1 e 2 gravados pelo caminho Python)
PERGUNTA = ("Explique em detalhes, em português, como funciona a quantização de modelos de linguagem: o que é um peso, "
            "por que 4 bits bastam na maior parte do modelo, o que se perde e como medir essa perda. Dê exemplos "
            "concretos e escreva pelo menos 1500 palavras, em seções.")

In [ ]:
import os, re, json, shutil, subprocess, time
# onde estamos: no Colab tudo fica em /content; no Kaggle os arquivos grandes vão para o disco com mais espaço (o
# /content vira um link para ele) e os resultados vão para /kaggle/working, que aparece na aba Output
KAGGLE = os.path.exists('/kaggle')
COLAB = os.path.exists('/content') and not KAGGLE
if not os.path.exists('/content'):
    cands = [d for d in ('/kaggle/tmp', '/kaggle/temp', '/tmp', '/root') if os.path.isdir(d)]
    BASE = os.path.join(max(cands, key=lambda d: shutil.disk_usage(d).free), 'strata_e6')
    os.makedirs(BASE, exist_ok=True)
    os.symlink(BASE, '/content')
SAIDA = '/kaggle/working' if KAGGLE else '/content'
PLATAFORMA = 'Kaggle' if KAGGLE else 'Colab' if COLAB else 'outra máquina'
N = os.cpu_count()
NUCLEOS = len({l.split(':')[1].strip() for l in open('/proc/cpuinfo') if l.startswith('core id')}) or N
# um thread por núcleo físico (os irmãos de HT ficam para o download); com 4 ou menos, usa todos
THREADS = N if N <= 4 else max(1, min(NUCLEOS, N))
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True)
GPU = gpu.stdout.strip().splitlines()[0] if gpu.returncode == 0 else ''
CPU = subprocess.run("lscpu | grep 'Model name' | cut -d: -f2", shell=True, capture_output=True, text=True).stdout.strip()
T_SUB = THREADS - 1 if T_SUBIDA < 0 else T_SUBIDA
DISCO = shutil.disk_usage(os.path.realpath('/content')).free / 1e9
print(f'{PLATAFORMA} | CPU: {CPU}, {N} threads, {NUCLEOS} núcleos físicos (geração com {THREADS}) | RAM {RAM:.0f} GB | GPU: {GPU or "nenhuma"} | disco livre {DISCO:.0f} GB')
if DISCO < 50:
    print(f'Aviso: o pico de disco é ~40 GB (BF16 16 GB + Q4_K_M 5 GB + arquivos de estrato 6 GB + GGUFs de nível 12 GB).')
if CONFERIR_BIT_A_BIT and not GPU:
    print('Sem GPU: a conferência bit a bit fica de fora (no CPU ela leva horas). O resto roda.')
    CONFERIR_BIT_A_BIT = False
def sh(cmd, mostra=True):
    r = subprocess.run("set -o pipefail; " + cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, executable="/bin/bash")
    if mostra:
        print(r.stdout[-3000:])
    if r.returncode:
        raise SystemExit(f"falhou: {cmd}")
    return r.stdout

In [ ]:
STRATA_SHA = "080a2fa3df4263602b83ccc3530ea625aff81a1b"   # o commit do STRATA com o E6
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
sh("test -f /usr/include/openssl/evp.h || (apt-get -qq update && apt-get -qq install -y libssl-dev) > /dev/null 2>&1; test -f /usr/include/openssl/evp.h", False)
sh("which cmake > /dev/null || pip install -q cmake", False)
sh(f"rm -rf strata && git init -q strata && cd strata && git remote add origin https://github.com/Cantyll/strata.git && git fetch -q --depth 1 origin {STRATA_SHA} && git checkout -q FETCH_HEAD && git log -1 --oneline")
sh(f"rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin {LLAMA_SHA} && git checkout -q FETCH_HEAD")
sh("sh strata/kernel/ggml/aplicar.sh llama.cpp | tail -2")
sh("pip install -q -U transformers safetensors huggingface_hub hf_xet && pip install -q -e llama.cpp/gguf-py", False)
# só CPU: a geração roda na CPU, com os pesos do jeito que estão no arquivo (sem reempacotar)
sh("cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=OFF -DGGML_NATIVE=ON > /dev/null && cmake --build build -j $(nproc) --target llama 2>&1 | tail -1")
L = '/content/llama.cpp'
sh(f"mkdir -p /content/b && cd /content/b && gcc -O2 -march=native -ffp-contract=off -c -I /content/strata/kernel /content/strata/kernel/strata_q4kr.c /content/strata/kernel/strata_q4kr_avx2.c /content/strata/kernel/strata_separado_avx2.c"
   f" && g++ -O2 -march=native -std=c++17 -I {L}/include -I {L}/ggml/include -I /content/strata/kernel /content/strata/tests/demanda/strata_subir.cpp *.o -L {L}/build/bin -lllama -lggml -lggml-base -Wl,-rpath,{L}/build/bin -lpthread -lcrypto -o strata_subir && ls -la strata_subir")
# o strata-ref (Rust) grava os arquivos de estrato
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null 2>&1")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
sh("cd strata/leitor-rs && cargo build --release -q 2>&1 | tail -3; ls -la target/release/strata-ref")
try:   # um token do HF é opcional (os dois repositórios são públicos)
    if COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    elif KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

O Q4_K_M do Hub (medindo o download de verdade), os pesos BF16 e os arquivos de estrato.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
t0 = time.time()
Q4 = hf_hub_download(GGUF_REPO, GGUF_ARQ, local_dir='/content/hub')
T_Q4 = time.time() - t0
B_Q4 = os.path.getsize(Q4)
print(f'Q4_K_M: {B_Q4 / 1e9:.2f} GB em {T_Q4:.0f} s ({8 * B_Q4 / T_Q4 / 1e6:.0f} Mbit/s)')
SRC = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"], local_dir='/content/hf')
shards = sorted(f'{SRC}/{f}' for f in os.listdir(SRC) if f.endswith('.safetensors'))
DIST = '/content/dist'
shutil.rmtree(DIST, ignore_errors=True)
t0 = time.time()
sh(f"strata/leitor-rs/target/release/strata-ref distribui {Q4} {DIST} {' '.join(shards)} --index {SRC}/model.safetensors.index.json --niveis 3 | tail -12")
T_DIST = time.time() - t0
sh(f"python strata/tests/demanda/estratos_idx.py {DIST}/model.strata.index.json {Q4} {DIST}/subida.idx")
ix = json.load(open(f'{DIST}/model.strata.index.json'))
ARQS = {a['nome']: a['bytes'] for a in ix['arquivos']}
B_PLANOS = sum(b for n, b in ARQS.items() if '-e01-' in n or '-e02-' in n)
print(f'distribui: {T_DIST:.0f} s; e01 + e02 = {B_PLANOS / 1e9:.2f} GB')

Conferência bit a bit: o modelo que sobe com os planos dos arquivos de estrato (modo `conferir`: 0 → 1 → 2 → 2 com o Q6_K separado → 1 → 0) contra os GGUFs de nível 1 e 2 gravados do zero pelo caminho Python, e contra o próprio Q4_K_M. 32 posições do wikitext, logits inteiros.

In [ ]:
import urllib.request, zipfile, io
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
SUB = '/content/b/strata_subir'
BIT = {}
VEL = {}
os.makedirs('/content/cf', exist_ok=True)
out = sh(f"cd /content/cf && {SUB} conferir {Q4} {DIST}/subida.idx /content/wiki.test.raw c -n 32 -t {THREADS} 2>&1 | grep -v '^[.]*$'")
for m in re.finditer(r'(c_n[0-9a-z_]+)\.bin: \d+ posições, ([0-9.]+) tokens/s', out):
    VEL[m.group(1)] = float(m.group(2))
m = re.search(r'conferidos pelo SHA-256 do índice: (\d+)', out)
CONFERIDOS_CF = int(m.group(1)) if m else 0
if CONFERIR_BIT_A_BIT:
    sh(f"cd /content/cf && {SUB} ref {Q4} - /content/wiki.test.raw r_q4km -n 32 -t {THREADS} 2>&1 | tail -1")
    # com disco de sobra, os dois níveis saem numa passada; com pouco, um de cada vez (um GGUF de nível por vez no disco)
    passadas = [[1, 2]] if shutil.disk_usage(os.path.realpath('/content')).free / 1e9 > 20 else [[1], [2]]
    for nv in passadas:
        outs = ' '.join(f'/content/n{k}.gguf' for k in nv)
        sh(f"python strata/tests/kernel/strata_gguf_niveis.py --model {SRC} --gguf {Q4} --nivel {' '.join(map(str, nv))} --out {outs} 2>&1 | grep -v Warning | tail -3")
        for k in nv:
            sh(f"cd /content/cf && {SUB} ref /content/n{k}.gguf - /content/wiki.test.raw r_n{k} -n 32 -t {THREADS} 2>&1 | tail -1")
            os.remove(f'/content/n{k}.gguf')
    for c, r in (('c_n0', 'r_q4km'), ('c_n1', 'r_n1'), ('c_n2', 'r_n2'), ('c_n2s', 'r_n2'), ('c_n1_desceu', 'r_n1'), ('c_n0_desceu', 'r_q4km')):
        BIT[c] = open(f'/content/cf/{c}.bin', 'rb').read() == open(f'/content/cf/{r}.bin', 'rb').read()
        print(c, '=' if BIT[c] else 'DIFERE de', r)
    sh("rm -f /content/n1.gguf /content/n2.gguf")
shutil.rmtree('/content/hf', ignore_errors=True)   # o BF16 não é mais usado
# tokens por rodada: o download dos dois níveis com folga, na velocidade do nível mais lento (a geração cede um
# núcleo durante o download, por isso a margem)
LENTO = min(v for k, v in VEL.items() if k in ('c_n0', 'c_n1', 'c_n2', 'c_n2s'))
for mbit in MBITS:
    if mbit not in N_TOKENS:
        N_TOKENS[mbit] = min(6000, int((B_PLANOS * 8 / (mbit * 1e6) * 1.4 + 60) * LENTO) + 60)
print(f'nível mais lento: {LENTO:.2f} tokens/s; tokens por rodada: {N_TOKENS}')

A subida com download de verdade. Para cada teto de banda: um servidor HTTP local serve a pasta da distribuição, o baixador começa no token 16 e o modelo (já respondendo pelo Q4_K_M) sobe de nível conforme os planos chegam. O controle é o mesmo download, sem subida (`--alvo 0`).

In [ ]:
os.makedirs('/content/r', exist_ok=True)
chat = ("<|start_header_id|>user<|end_header_id|>\n\n" + PERGUNTA + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n")
open('/content/r/prompt.txt', 'w').write(chat)
RES = {}
for mbit in MBITS:
    porta = 8700 + mbit
    srv = subprocess.Popen(['python3', 'strata/tests/dist/servidor_faixas.py', DIST, '--porta', str(porta), '--mbit', str(mbit)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    for alvo in (2, 0):
        D = f'/content/r/baixa_{mbit}_{alvo}'
        shutil.rmtree(D, ignore_errors=True)
        os.makedirs(D)
        shutil.copy(f'{DIST}/subida.idx', D)
        nome = f'm{mbit}_a{alvo}'
        baixar = f'python3 /content/strata/tests/dist/baixa_estratos.py http://127.0.0.1:{porta} {D} --ate-nivel 2 --conexoes 8 --pedaco-mb 16'
        t0 = time.time()
        out = sh(f"{SUB} gerar {Q4} {D}/subida.idx /content/r/prompt.txt /content/r/{nome}.csv -t {THREADS} -n {N_TOKENS[mbit]} --inicio {INICIO} "
                 f"--alvo {alvo} --orcamento-ms 4 --t-subida {T_SUB} --chat 1 --espera 60 --baixar '{baixar}' 2>&1 | grep -v '^[.]*$'")
        open(f'/content/r/{nome}.log', 'w').write(out)
        RES[nome] = {'saida': out, 'segundos_total': time.time() - t0}
        shutil.rmtree(D, ignore_errors=True)
    srv.terminate()

Resumo e arquivos para a thread.

In [ ]:
import csv, statistics as st
from IPython.display import Markdown, display
def num(pad, txt, g=1):
    m = re.search(pad, txt)
    return float(m.group(g)) if m else None
def tokens(nome):
    r = list(csv.DictReader(open(f'/content/r/{nome}.csv')))
    return [1000 * float(x['segundos']) for x in r], [int(x['trocas']) for x in r], r
def faixa(t, a, b):
    d = sorted(t[a:b + 1])
    return st.median(d), d[max(0, int(.99 * len(d)) - 1)], d[-1]
T = ['# STRATA, E6: o pacote inteiro no Llama-3.1-8B-Instruct', '',
     f'Máquina ({PLATAFORMA}): {CPU}, {N} threads ({NUCLEOS} núcleos físicos; geração com {THREADS} threads, {T_SUB} enquanto o download corre), RAM {RAM:.0f} GB. '
     f'Nível 0: `{GGUF_REPO}/{GGUF_ARQ}` ({B_Q4 / 1e9:.2f} GB, baixado do Hub em {T_Q4:.0f} s, {8 * B_Q4 / T_Q4 / 1e6:.0f} Mbit/s). '
     f'Arquivos de estrato gravados pelo `strata-ref distribui` em {T_DIST:.0f} s; os planos dos níveis 1 e 2 (e01 + e02) somam {B_PLANOS / 1e9:.2f} GB.', '',
     '## Conferência bit a bit (32 posições, logits inteiros)', '']
if BIT:
    T += ['| modelo que subiu (arquivos de estrato) | contra | igual bit a bit? |', '|---|---|---|']
    ref = {'c_n0': 'Q4_K_M', 'c_n1': 'GGUF nível 1', 'c_n2': 'GGUF nível 2', 'c_n2s': 'GGUF nível 2', 'c_n1_desceu': 'GGUF nível 1', 'c_n0_desceu': 'Q4_K_M'}
    nomes = {'c_n0': 'nível 0', 'c_n1': 'nível 1', 'c_n2': 'nível 2', 'c_n2s': 'nível 2 com o Q6_K separado (S2)', 'c_n1_desceu': 'desceu para o 1', 'c_n0_desceu': 'desceu para o 0'}
    for c, ok in BIT.items():
        T.append(f'| {nomes[c]} | {ref[c]} | {"**sim**" if ok else "**NÃO**"} |')
else:
    T.append('Não rodou (sem GPU).')
T += ['', f'Planos conferidos pelo SHA-256 do índice na conferência: {CONFERIDOS_CF}.', '',
      '## Velocidade de cada nível (modo conferir, 32 posições, um token por vez)', '', '| nível | tokens/s |', '|---|---:|']
for c, v in VEL.items():
    T.append(f'| {c} | {v:.2f} |')
T += ['', '## Subida com download de verdade', '',
      '| teto | 1º token (carga + prompt) | download dos planos | nível 1 completo | nível 2 completo | Q6_K montados (P2) | planos conferidos | tok/s antes | tok/s durante |',
      '|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
RESUMO = {}
for mbit in MBITS:
    o = RES[f'm{mbit}_a2']['saida']
    t, tr, r = tokens(f'm{mbit}_a2')
    tc, _, _ = tokens(f'm{mbit}_a0')
    a = next(i for i, x in enumerate(tr) if x) if any(tr) else INICIO
    b = max(i for i, x in enumerate(tr) if x) if any(tr) else len(t) - 9
    base = st.median(t[:a])
    p50, p99, mx = faixa(t, a, b)
    c50, c99, cmx = faixa(tc, a, b)
    x = {'primeiro_token_s': num(r'1º token aos ([0-9.]+) s', o), 'baixador_s': num(r'baixador: saída \d+, terminou em ([0-9.]+) s', o),
         'nivel1_s': num(r'nível 1 completo em (-?[0-9.]+) s', o), 'nivel2_s': num(r'nível 2 em (-?[0-9.]+) s', o),
         'p2_s': num(r'montados em P2 em (-?[0-9.]+) s', o), 'conferidos': num(r'conferidos pelo SHA-256 do índice: (\d+)', o),
         'tok_s_antes': 1000 / base, 'tok_s_durante': 1000 * (b - a + 1) / sum(t[a:b + 1]),
         'janela': [a, b], 'subida': {'p50': p50, 'p99': p99, 'max': mx}, 'controle': {'p50': c50, 'p99': c99, 'max': cmx},
         'base_ms': base}
    RESUMO[mbit] = x
    f = lambda v: f'{v:.1f} s' if v is not None and v >= 0 else '—'
    T.append(f"| {mbit} Mbit/s | {f(x['primeiro_token_s'])} | {f(x['baixador_s'])} | {f(x['nivel1_s'])} | {f(x['nivel2_s'])} | {f(x['p2_s'])} | "
             f"{int(x['conferidos'] or 0)} | {x['tok_s_antes']:.2f} | {x['tok_s_durante']:.2f} |")
T += ['', 'Tempos da subida contados a partir do token 16, quando o download começa.', '',
      '## Pausa: a mesma faixa de tokens, com subida e no controle (mesmo download, sem subida)', '',
      '| teto | tokens na faixa | p50 subida / controle (ms) | p99 subida / controle (ms) | pior token subida / controle (ms) | pior token em tokens normais | regra (p99 e pior ≤ 1,5× o controle) |',
      '|---:|---:|---:|---:|---:|---:|---|']
for mbit, x in RESUMO.items():
    s_, c_ = x['subida'], x['controle']
    ok = s_['p99'] <= 1.5 * c_['p99'] and s_['max'] <= 1.5 * c_['max']
    T.append(f"| {mbit} Mbit/s | {x['janela'][1] - x['janela'][0] + 1} | {s_['p50']:.0f} / {c_['p50']:.0f} | {s_['p99']:.0f} / {c_['p99']:.0f} | "
             f"{s_['max']:.0f} / {c_['max']:.0f} | {s_['max'] / x['base_ms']:.1f} | {'passa' if ok else 'não passa'} |")
T += ['', 'O p50 durante a subida é maior também porque os níveis 1 e 2 leem mais bytes por token; a pausa é o p99 e o pior token.', '',
      '## A resposta (100 Mbit/s), com as marcas das trocas', '']
txt = open(f'/content/r/m{MBITS[0]}_a2.csv.txt').read()
T += ['```text', txt[:6000] + ('\n[...]' if len(txt) > 6000 else ''), '```']
rel = '\n'.join(T)
open(f'{SAIDA}/strata_8b_e6.md', 'w').write(rel)
json.dump({'plataforma': PLATAFORMA, 'n_tokens': N_TOKENS, 'cpu': CPU, 'threads': THREADS, 't_subida': T_SUB, 'q4km_bytes': B_Q4, 'q4km_download_s': T_Q4, 'distribui_s': T_DIST, 'planos_bytes': B_PLANOS,
           'bit_a_bit': BIT, 'tok_s_conferir': VEL, 'conferidos_conferir': CONFERIDOS_CF, 'subida': RESUMO,
           'logs': {k: v['saida'][-4000:] for k, v in RES.items()}}, open(f'{SAIDA}/strata_8b_e6.json', 'w'), indent=1)
import glob
with zipfile.ZipFile(f'{SAIDA}/strata_8b_e6_csv.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(glob.glob('/content/r/*.csv') + glob.glob('/content/r/*.txt') + glob.glob('/content/r/*.log')):
        zf.write(f, os.path.basename(f))
ARQS_SAIDA = [f'{SAIDA}/strata_8b_e6.md', f'{SAIDA}/strata_8b_e6.json', f'{SAIDA}/strata_8b_e6_csv.zip']
for f in ARQS_SAIDA:
    print(f, os.path.getsize(f), 'bytes')
display(Markdown(rel))

In [ ]:
if COLAB:
    from google.colab import files
    for f in ARQS_SAIDA:
        files.download(f)
else:
    print('Os arquivos estão em', SAIDA, '(no Kaggle: aba Output do painel da direita, ou o menu de cada arquivo em /kaggle/working):')
    for f in ARQS_SAIDA:
        print('  ', f)